# RLVR PII Masking

This notebook builds the redaction environment used by the project. It follows the linked `rl-for-llms` task setup: examples come from `AdamLucek/open-pii-masking-en-us-30k`, private spans are replaced with the generic `[PII]` token, and model completions are parsed from `<masked_output>` XML tags.

The task is a good fit for RLVR because it has direct checks. We can compare the parsed output with the reference answer, count how many `[PII]` masks were produced, verify the XML wrapper, and check that normal text was not unnecessarily removed.

In [ ]:
%pip install -q -e ..[train]

In [ ]:
from redaction_rlvr.data import DATASET_NAME, load_pii_dataset
from redaction_rlvr.rewards import RedactionReward, format_completion, extract_masked_output

train_dataset, eval_dataset = load_pii_dataset(num_train_examples=64, num_eval_examples=16, seed=42)
DATASET_NAME, train_dataset[0]

Each row keeps the original question, the expected masked answer, and metadata. The `pii_count` value is used as a partial reward signal when exact text matching is too sparse.

In [ ]:
example = train_dataset[0]
print(example["question"])
print(example["answer"])
print(example["info"])

The reward is decomposed into four verifiable pieces: redaction accuracy, output-format correctness, mask-count correctness, and preservation of non-sensitive content.

In [ ]:
reward = RedactionReward()
perfect_completion = format_completion(example["answer"])
breakdown = reward.breakdown(perfect_completion, example["answer"], info=example["info"])
breakdown, breakdown.total, extract_masked_output(perfect_completion)

A weaker completion may still receive partial credit. For instance, a model can use the right XML wrapper and produce the right number of masks while failing the exact reference string.

In [ ]:
rough_completion = format_completion("[PII] [PII]")
reward.breakdown(rough_completion, example["answer"], info=example["info"])

Training uses TRL's GRPO trainer. The project script passes dataset columns to the reward function, so each generated completion is scored against the row's reference answer and metadata.

In [ ]:
!python ../scripts/train_rlvr_redactor.py \
  --model Qwen/Qwen3-4B-Instruct-2507 \
  --output-dir ../outputs/redaction-rlvr \
  --num-train-examples 128 \
  --num-eval-examples 32 \
  --max-steps 20 \
  --layer-mode all